

### Stochastic Processes

A stochastic process is a collection of random variables $\{X_{t}\}$ where $t \in T$. The random variables can be discrete or continuous and the index set $T$ can similiarly be discrete or continuous. To start we will consider stochastic processes with a discrete index set $T = \{0,1,2,\dots\}$ and 2 possible states. 

Therefore $X_t$ will take values in $\mathcal{X} = \{a,b\}$ and its index set $T$ will be $\{0,1,\dots\}$. 

### Markov Chains 

To simplify our stochastic process we will make a memorylessness assumption. That is, the probability $X_{t+1} = x$ can be determined from just the current state of the chain $X_t$. Formally, 

$$
P(X_{t+1} = x| X_{0} = x_0, X_{1} = x_1,\dots,X_{t} = x_t) = P(X_{t+1} = x|X_{t} = x_{t}), x_{t} \in \mathcal{X}. 
$$

This is called a Markov chain and is one of the central objects in the study of stochastic processes. 

### Example

Let's consider a motivating example of a Markov chain. Let $X_{t}$ be the weather in Flagstaff on a given day $t$. The value of $X_{t}$ can be one of two states $\{\text{sun},\text{rain}\}$. To finish off our definiton of $X_t$ we will need a transition matrix $T$. 

$$
T = \begin{bmatrix} 0.7 & 0.3 \\ 0.3 & 0.7 \end{bmatrix}
$$

The entries $T_{ij}$ represent the probability of moving from state $i$ to state $j$. For example $T_{00} = 0.7$ says that given it is sunny today the probability it will be sunny tomorrow is $0.7$. 

Our memoryless assumption ensures that we can write $P(X_{t+1} = j | X_{t} = i) = T_{ij}$ using the matrix $T$. 

We will write the probability of $X_t$ as 

$$
P(X_{t}) = [P(\text{sun}),P(\text{cloud})]^{T}.
$$

For example suppose on day $0$ it is cloudy. Then $P(X_{0}) = [0.0,1.0]^{T}$. We know the weather on day 0 is cloudy so $P(X_{0} = \text{cloud}) = 1.0$. 

A natural next question arises, what is $P(X_{1}|X_{0} = x_0)$? Given our transition matrix $T$ we can compute 

$$
P(X_{1}|X_{0} = x_0) = P(X_{0}) \cdot T
$$

by matrix multiplication. 

Lets code this up in python. 


In [32]:
import numpy as np

PX_0 = np.array([0.0,1.0])

T = np.array([[0.7,0.3],
              [0.3,0.7]])

PX_1 = PX_0.T @T #The @ sign denotes matrix multiplication in numpy

PX_1

array([0.3, 0.7])

In fact, given any starting state $P(X_0)$ we can compute $P(X_t)$ as

$P(X_t|X_0 = x_0) = P(X_{0}) \cdot T^t$.

Alternatively, we can compute $P(X_t|X_{0})$ recursively by iterating through computations of the intermediate states. 

In [ ]:
PX_0 = np.array([0.0,1.0])

T = np.array([[0.7,0.3],
              [0.3,0.7]])

t = 3

PX_t = PX_0.T @ np.linalg.matrix_power(T,t) 

PX_t

array([0.468, 0.532])

In [34]:
PX = np.array([0.0,1.0])

T = np.array([[0.7,0.3],
              [0.3,0.7]])

t = 3

for i in range(t):
    PX = PX.T @ T

PX


array([0.468, 0.532])

The two techniques for computing $P(X_{t}|X_{0} = x_0)$ yield equivalent results. 

Next, lets consider what happens if we keep iterating. 

In [38]:
PX_0 = np.array([0.0,1.0])

T = np.array([[0.7,0.3],
              [0.3,0.7]])

ts = [2,5,10,100]

for t in ts: 
    PX_t = PX_0.T @ np.linalg.matrix_power(T,t) 
    print(f"PX_t|X_0 at time {t} = {PX_t}")


PX_t|X_0 at time 2 = [0.42 0.58]
PX_t|X_0 at time 5 = [0.49488 0.50512]
PX_t|X_0 at time 10 = [0.49994757 0.50005243]
PX_t|X_0 at time 100 = [0.5 0.5]


It appears $P(X_t|X_0 = x_0)$ is approaching a constant vector $[0.5,0.5]^T$. Is this just because of our specific choice of $x_0 = [0.0,1.0]^T$? 
What if we choose $x_0 = [1.0,0.0]^T$?

In [39]:
PX_0 = np.array([1.0,0.0])

T = np.array([[0.7,0.3],
              [0.3,0.7]])

ts = [2,5,10,100]

for t in ts: 
    PX_t = PX_0.T @ np.linalg.matrix_power(T,t) 
    print(f"PX_t|X_0 at time {t} = {PX_t}")


PX_t|X_0 at time 2 = [0.58 0.42]
PX_t|X_0 at time 5 = [0.50512 0.49488]
PX_t|X_0 at time 10 = [0.50005243 0.49994757]
PX_t|X_0 at time 100 = [0.5 0.5]


It turns out, for any choice of $x_0$, $P(X_t|X_0 = x_0)$ will approach a constant vector. 
Formally we can write 

$\lim\limits_{t\rightarrow \infty} P(X_t) = [0.5,0.5]^T$. This means that for any initial condition $x_0$ the distribution of $X_t$ will eventually approach a uniform distribution with a 50% chance of sun and a 50% chance of clouds. 

This is called the stationary distribution of the Markov chain. A Markov chain will approach a stationary distribution if two properties are satisfied, aperiodicity and irreducibility. 

### Aperiodicity

A state $i$ of a Markov chain has a period 

$$
d(i) = \text{gcd}\{t \geq 1 : P(X_{t} = i|X_{0} = i) > 0\}
$$. 

In our example, let $i = \text{sun}$ . Then 
$$

d(\text{sun}) = \text{gcd} \{t \geq 1 : P(X_{t} = \text{sun}|X_{0} = \text{sun}) > 0\}  = 1

$$ 

similarly, $d(\text{cloud}) = 1$. As all states in the Markov chain have period $1$ we say the chain is aperiodic. 

### Irreducibility

A Markov chain in which every state can be reached from every other state is called an irrducible Markov chain. A state $j$ is said to be accessible(denoted $i \rightarrow j$) from a state $i$ if there exists some $t \geq 0$ such that 

$$
P(X_{t} = j|X_{0} = i) > 0.
$$

If $i \rightarrow j$ and $j \rightarrow i$ then state $i$ and $j$ are said to communicate. If all pairs of states communicate then the chain is irreducible. 

Our example chain is therefore clearly irreducible. 

### Stationary Distribution

As our example chain $X_{t}$ is both aperiodic and irreducible we conclude it has a stationary distribution $P_{\infty}(X_t)$. 

To find the stationary distribution algebraically we note the stationary distribution must statisfy the equation 

$$
P_{\infty}(X_t) \cdot T = P_{\infty}(X_t)
$$
together with $P_{\infty}(\text{sun}) + P_{\infty}(\text{cloud}) = 1$. 

First, let's expand $P_{\infty}(X_t)$ as $[p_0,p_1]$. Then we can write our equations as 

$$
[p_1,p_2]^T \cdot \begin{bmatrix} 0.7 & 0.3 \\ 0.3 & 0.7 \end{bmatrix}  = [p_1,p_2] \\

p_1+p_2 = 1
$$

expanding yields two equations 

$$
p_2 = p_1 \\
p_1 + p_2 = 1
$$

therefore $p_1 = p_2 = 0.5$. So the stationary distribution is $[0.5,0.5]^T$.


### Detailed Balance

To derive the Metropolis Hastings algorithm we utilize an even stronger condition than aperiodicity and irreducibility which also implies the existence of a stationary distribution. 

A Markov chain satisfies detailed balance if there is a stationary distribution that satisfies the detailed balance equations. 

$$
\pi_i T_{ij} = \pi_j T_{ji}. 
$$

That is the probability of going from state $i \rightarrow j$ is equal to $j \rightarrow i$ in the stationary distribution. As our transition matrix $T$ is symmetric this chain satisfies the detailed balance equations. 

### Metropolis-Hastings

The Metropolis-Hastings algorithm is an algorithm for sampling from a probability distribution with a possibly unknown normalization constant. This kind of problem arises quite frequently in Bayesian inference and Metropolis-Hastings is the bedrock algorithm of computational Bayesian inference. 